Most of this work (code and explanations) are from Shashank Kapadia, you can find his work [here](https://towardsdatascience.com/end-to-end-topic-modeling-in-python-latent-dirichlet-allocation-lda-35ce4ed6b3e0)!

I just want to show it applied to our subject!

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Open the train dataset
train_df = pd.read_csv('../input/actuarial-loss-estimation/train.csv')

In [ ]:
# Selecting some columns to work with
Claim_description = train_df[['ClaimNumber', 'DateTimeOfAccident', 'ClaimDescription', 'InitialIncurredCalimsCost', 'UltimateIncurredClaimCost']].copy()

In [ ]:
# Converting the date data to year only
Claim_description['DateTimeOfAccident'] = pd.to_datetime(Claim_description['DateTimeOfAccident'])
Claim_description['DateTimeOfAccident'] = Claim_description['DateTimeOfAccident'].dt.year

In [ ]:
Claim_description

This work has been done following [Shashank Kapadia's tutorial](https://towardsdatascience.com/end-to-end-topic-modeling-in-python-latent-dirichlet-allocation-lda-35ce4ed6b3e0) on towards data science.

The code is based on his work, I paramaetrized the code, modified comments so that it sticks to our dataset and added new comments to better understand the process from my data science level.

## Exploratory Analysis
We are going to make a word cloud to get a visual representation of most common words. It is an important step to verify if any preprocessing is necessary before training the model.

In [ ]:
# Import the wordcloud library
from wordcloud import WordCloud
# Join the different descriptions together
long_string = ','.join(list(Claim_description['ClaimDescription'].values))
# Create a WordCloud object
wordcloud = WordCloud(background_color="white", max_words=5000, contour_width=3, contour_color='steelblue')
# Generate a word cloud
wordcloud.generate(long_string)
# Visualize the word cloud
wordcloud.to_image()


The information we are getting are already meaningfull. 
We can make the hypothesis that two really different pain kind, like 'left hand' and 'back strain' will have a different impact on our target (ultimate claim cost).

## Prepare data for LDA Analysis
We are going to transform the textual data in a format that will serve as an input for training LDA model. LDA means Latent Dirichlet Allocation, it's a "topic modeling technique that assumes each topic is a mixture over an underlying set of words and each document is a mixture of a set of topic probabilities".

We start by tokenizing the text : 
> Splitting each description into smaller units (words and ponctuation) that are called tokens.

Then, we remove the stopwords : 
> Stopwords are words that are so common that it would be useless to use it in our model. EX : Can, I, ponctuation ....


In [ ]:
# Importing libraries
import gensim # open-source library for unsupervised topic modeling and NLP
from gensim.utils import simple_preprocess
import nltk # Natural Language Toolkit is a library to work with human language data

# Storing a list of stopwords
nltk.download('stopwords') # start the NLTK Downloader and download the stopwords
from nltk.corpus import stopwords 
stop_words = stopwords.words('english') # Selecting english stopwords
stop_words.extend(['from', 'subject', 're', 'edu', 'use']) # adding new stopwords

# Defining a function to convert our descriptions to word tokens and remove ponctuation
def sent_to_words(descriptions):
    i = 0
    for description in descriptions:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(description), deacc=True)) 
        
# Defining a function to remove stopwords    
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]
        
data = Claim_description.ClaimDescription.values.tolist()
data_words = list(sent_to_words(data))
data_words = remove_stopwords(data_words)

print(data_words[0]) #Showing our first description tokens

Next, we convert the tokenized object into a corpus and dictionary.
> A Corpus in genism is a collection of documents (descriptions in our case).

> A Dictionary encapsulates the mapping between normalized words and their integer ids.

In [ ]:
import gensim.corpora as corpora
# Create Dictionary
id2word = corpora.Dictionary(data_words)
# Create Corpus
texts = data_words

# Term Document Frequency
# Converts a collection of words to  a list of (word_id, word_frequency) 2-tuples.
corpus = [id2word.doc2bow(text) for text in texts]

# View first description
print(corpus[0])

## LDA model training with default parameters
We will start to build a model with 10 topics where each topic is a combination of keywords, and each keyword contributes a certain weight to the topic.

In [ ]:
from pprint import pprint
# number of topics
num_topics = 10
# Build LDA model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=num_topics,
                                       random_state=0)
# Print the Keyword in the 10 topics
pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

## Analyzing LDA model results
Now that we have a trained model let’s visualize the topics for interpretability. To do so, we’ll use a popular visualization package, pyLDAvis which is designed to help interactively with:

1. Better understanding and interpreting individual topics, and
2. Better understanding the relationships between the topics.

For (1), you can manually select each topic to view its top most frequent and/or “relevant” terms, using different values of the λ parameter. This can help when you’re trying to assign a human interpretable name or “meaning” to each topic.

For (2), exploring the Intertopic Distance Plot can help you learn about how topics relate to each other, including potential higher-level structure between groups of topics.

In [ ]:
import pyLDAvis.gensim
import pickle 
import pyLDAvis

In [ ]:
pyLDAvis.enable_notebook()
LDAvis_data_filepath = os.path.join('./ldavis_prepared_'+str(num_topics))

In [ ]:
LDAvis_data_filepath

In [ ]:
# # this is a bit time consuming - make the if statement True
# # if you want to execute visualization prep yourself
if 1 == 1:
    LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
    with open(LDAvis_data_filepath, 'wb') as f:
        pickle.dump(LDAvis_prepared, f)

In [ ]:
# load the pre-prepared pyLDAvis data from disk
with open(LDAvis_data_filepath, 'rb') as f:
    LDAvis_prepared = pickle.load(f)
pyLDAvis.save_html(LDAvis_prepared, './ldavis_prepared_'+ str(num_topics) +'.html')
LDAvis_prepared

We already have quite good results without tweaking the parameters of the model!
* Topic 1: Hand Injuries
* Topic 2: Back Pains
* Topic 3: Mixed topics
* Topic 4: Mixed topics but looks like Topic 3 with right keyword
* Topic 5: Back Pains
* Topic 6: Mixed pain kind
* Topic 7: Really mixed topic
* Topic 8: Hand Injuries
* Topic 9: Little category
* Topic 10: Little category

Those topics start to make sense, but we will have to tweak our parameters: 
* Lots of topics mainly contain right or left as first words, while it should be a topic on its own or be considered as stop words not to pollute our data. Since about 90% of the population is right-handed, it would be interesting to keep this in the data.
* Lots of topics are made of several topics and should be divided.
* In our dataset it would be interesting to separate the body member from the kind of pain, thus we will need more topics 


## New try without left and right keywords

In the future, we can add them back in a new column of our dataset with a simple if statement.

In [ ]:
stop_words.extend(['left', 'right']) # adding new stopwords

     
data_wrl = Claim_description.ClaimDescription.values.tolist()
data_words_wrl = list(sent_to_words(data_wrl))
data_words_wrl = remove_stopwords(data_words_wrl)

# Create Dictionary
id2word = corpora.Dictionary(data_words_wrl)
# Create Corpus
texts = data_words_wrl

# Term Document Frequency
# Converts a collection of words to  a list of (word_id, word_frequency) 2-tuples.
corpus = [id2word.doc2bow(text) for text in texts]

# View first description
print(data_words_wrl[0])
print(corpus[0])

```
Now right and left disappeared:
['lifting', 'tyre', 'injury', 'right', 'arm', 'wrist', 'injury']
is now:
['lifting', 'tyre', 'injury', 'arm', 'wrist', 'injury']
```


In [ ]:
# number of topics
num_topics = 15
# Build LDA model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=num_topics,
                                           random_state=0)
# Print the Keyword in the 10 topics
pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

pyLDAvis.enable_notebook()
LDAvis_data_filepath = os.path.join('./ldavis_prepared_'+str(num_topics))

# # this is a bit time consuming - make the if statement True
# # if you want to execute visualization prep yourself
if 1 == 1:
    LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
    with open(LDAvis_data_filepath, 'wb') as f:
        pickle.dump(LDAvis_prepared, f)

# load the pre-prepared pyLDAvis data from disk
with open(LDAvis_data_filepath, 'rb') as f:
    LDAvis_prepared = pickle.load(f)
pyLDAvis.save_html(LDAvis_prepared, './ldavis_prepared_'+ str(num_topics) +'.html')
LDAvis_prepared      


This is already way better!

## Outputting the dominant topic for each text

In [ ]:
# Code from https://www.machinelearningplus.com/nlp/topic-modeling-gensim-python/#18dominanttopicineachsentence
# This may take several minutes to run, don't panick

def format_topics_sentences(ldamodel=lda_model, corpus=corpus, texts=data):
    # Init output
    sent_topics_df = pd.DataFrame()
    
    # Get main topic in each document
    for i, row in enumerate(ldamodel[corpus]):
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        # Get the Dominant topic, Perc Contribution and Keywords for each document
        for j, (topic_num, prop_topic) in enumerate(row):
            if j == 0:  # => dominant topic
                wp = ldamodel.show_topic(topic_num)
                topic_keywords = ", ".join([word for word, prop in wp])
                sent_topics_df = sent_topics_df.append(pd.Series([int(topic_num), round(prop_topic,4), topic_keywords]), ignore_index=True)
            else:
                break
    sent_topics_df.columns = ['Dominant_Topic', 'Perc_Contribution', 'Topic_Keywords']

    # Add original text to the end of the output
    contents = pd.Series(texts)
    sent_topics_df = pd.concat([sent_topics_df, contents], axis=1)
    return(sent_topics_df)


df_topic_sents_keywords = format_topics_sentences(ldamodel=lda_model, corpus=corpus, texts=data)

# Format
df_dominant_topic = df_topic_sents_keywords.reset_index()
df_dominant_topic.columns = ['Document_No', 'Dominant_Topic', 'Topic_Perc_Contrib', 'Keywords', 'Text']

# Show
df_dominant_topic.head(20)

I would like to remind that most of this work (code and explanations) are from Shashank Kapadia go check his work [here](https://towardsdatascience.com/end-to-end-topic-modeling-in-python-latent-dirichlet-allocation-lda-35ce4ed6b3e0)!

I just wanted to show it applied to our subject!